<a href="https://colab.research.google.com/github/aa4758/ML_DL_study/blob/master/Self%20Study%20ML%20%2B%20DL/CrossValidation%26GreedSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# hyperparameter: parameter that cannot trained by model is needed to set by user
# each model has their own hyperparameter and default value but user should set the hyperparameter to get the best result
# to get a best result, we have to test as mush as possible value, it makes the model getting fit to test set
# if we don't use test model, we cannot judge the model is overfitting or underfitting
# to prevent these problems, we make another training set called validation set
# training set for train the model with trying various hyperparameter value, test those tried model with validation set, using test set after find the best hyperparameter
# train set 60%, validation set 20%, test set 20%

In [4]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine-date')

In [6]:
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

In [8]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size = 0.2, random_state=42)         # train_set 80%, test_set 20%

In [9]:
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size = 0.2, random_state=42)    # train_set 60%, validation_set 20%

In [11]:
print(train_input.shape, sub_input.shape, val_input.shape)

(5197, 3) (4157, 3) (1040, 3)


In [12]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))          # overfitting, need to change hyperparameter and find better model
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


In [14]:
# we lose train_set by made validation set, 5197 to 4157
# many data, better model
# cross validation: devide a training set, repeat to set one of the training set as a validation set, train a model and test it with the validation set, score it, and find an average of the scores.
# k-fold cross validation: devide a training set by k
# default is 3 fold cross validation but mostly use 5 or 10 fold cross validation which can use 80 ~ 90% of data to train
# able to get stable score with less validation set

In [17]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
print(scores)
# fit_time: spent time to trian model
# score_time: spent time to validate model
# test_score: each fold validation score     != test score, the name is test score but it is not a test score, it's a validation score of each fold

{'fit_time': array([0.01346803, 0.01262021, 0.0132637 , 0.0128603 , 0.01250076]), 'score_time': array([0.0018394 , 0.00193787, 0.0019114 , 0.00235176, 0.00200796]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [19]:
import numpy as np
print(np.mean(scores['test_score']))

0.855300214703487


In [20]:
# cross_validate() does not shuffle the training set to devide fold
# in this case, we shuffled the whole training set by train_test-split so we didn't have to shuffle the data
# if u want to mix the data and cross validation at the same time, have to fix 'splitter'

In [25]:
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())        # use KFold splitter from regression model, StratifiedKFold splitter from classification model
print(np.mean(scores['test_score']))

0.855300214703487


In [28]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)        # `n_splits=10` : 10 fold cross validation, `shuffle=True` : shuffle data before to split (default is False)
scores = cross_validate(dt, train_input, train_target, cv=splitter)           # same way KFold class
print(np.mean(scores['test_score']))

0.8574181117533719


In [30]:
# hyperparameter tunning by keep changing hyperparameter and score it with cross validation
# each model has 1~6 hyperparameters
# all hyperparameters have to tuned at the same time
# we can do it with Grid Search which search hyperparameter and cross validation at the same time which means we do not have to call cross_validate()

In [35]:
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [36]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
# GridSearchCV(model, hyperparameters and its searching range, n_jobs)
# defaul of cv hyperparameter is 5.
# we have 5 hyperparameter and 5 fold cross validation, so train 5 * 5 = 25 models.
# n_jobs: number of CPU core to use parallel execution, default is 1, all the core from system is -1

In [37]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [38]:
# after to find a best hyperparameter, we have to train a model with the hyperparameter and all of training set
# GridSearch trains a model again with hyperparameter combination has highest validation score from 25 trained models.
# trained model is saved in best_estimator_ attributrion from gs object.
# best hyperparameter is saved in best_params_ attribution
# average scire of cross validation of each hyperparameter is saved in main_test_score key of cv_results_ attribution

In [39]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [41]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [43]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [45]:
best_index = np.argmax(gs.cv_results_['mean_test_score'])
print(gs.cv_results_['params'][best_index])

{'min_impurity_decrease': 0.0001}


In [49]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),       # 9
          'max_depth': range(5, 20, 1),                                     # 15
          'min_samples_split': range(2, 100, 10)                            # 10
          }

# range() is same with np.arange() but only for integer
# 9 * 15 * 10 = 1350 cross validations with these hyperparameter
# 1350 * 5 models with 5 fold cross validation

In [50]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)           # suffle data set ==> 5 fold cross validation
gs.fit(train_input, train_target)                                                       # ==> train a model with best hyperparameters combination and all of the data set

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [51]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [52]:
print(np.max(gs.cv_results_['mean_test_score']))        # best hyperparameter combination validation score

0.8683865773302731


In [53]:
print(gs.score(train_input, train_target))              # model test score

0.892053107562055


In [54]:
# Random Search: give standard deviation object sampling hyperparameters instead of list of hyperparameters value

In [56]:
from scipy.stats import uniform, randint
# sampling from continuous uniform distribution: pick the sample from given range uniformly
# randint: sampling integer
# uniform: sampling float

In [57]:
rgen = randint(0, 10)
rgen.rvs(10)          # randomly sampling

array([1, 7, 1, 1, 2, 3, 1, 5, 0, 9])

In [58]:
rgen.rvs(10)

array([5, 9, 9, 0, 4, 2, 2, 5, 4, 1])

In [59]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([101,  96,  99, 100,  82, 107, 117, 111, 102,  85]))

In [60]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.7196932 , 0.25356892, 0.76319172, 0.45236208, 0.02416334,
       0.06441824, 0.88593064, 0.57825412, 0.67056588, 0.07149978])

In [63]:
parms = {'min_impurity_decrease': uniform(0.0001, 0.001),
         'max_depth': randint(20, 50),
         'min_sample_split': randint(2, 25),
         'min_sample_leaf': randint(1, 25)
         }

In [64]:
from sklearn.model_selection import RandomizedSearchCV
gs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter = 100, n_jobs=-1, random_state=42)          # `n_iter=100`: sampling 100 times
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': range(5, 20),
                                        'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                                        'min_samples_split': range(2, 100, 10)},
                   random_state=42)

In [65]:
print(gs.best_params_)

{'min_samples_split': 12, 'min_impurity_decrease': np.float64(0.0005), 'max_depth': 11}


In [67]:
print(np.max(gs.cv_results_['mean_test_score']))

0.8681935292811135


In [70]:
dt = gs.best_estimator_                       # best model trained train_input and train_target (all training set) and saved in best_estimator_ attribution
print(dt.score(test_input, test_target))

0.8615384615384616
